In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

# Directorios de Entrada y Salida
DATA_DIR = Path('data')
OUTPUT_DIR = Path('datasets_procesados')

# Crear la carpeta de salida si no existe
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def limpiar_y_estandarizar_df(df):
    """
    Aplica reglas de limpieza estructural estricta:
    - Elimina filas vacías.
    - Limpia espacios en blanco en columnas y textos.
    - Convierte booleanos a enteros/texto limpio.
    """
    if df.empty:
        return df

    # 1. Eliminar filas completamente vacías
    df = df.dropna(how='all')

    # 2. Limpiar espacios en blanco en los nombres de las columnas
    df.columns = [str(col).strip() for col in df.columns]

    # 3. Limpiar celdas de texto (quitar espacios sobrantes al inicio y final)
    for col in df.select_dtypes(include=['object', 'string']).columns:
        df[col] = df[col].astype(str).str.strip()

    # 4. Convertir columnas booleanas (True/False) a enteros (1/0) o cadenas estandarizadas
    for col in df.select_dtypes(include=['bool']).columns:
        df[col] = df[col].astype(int)

    # 5. Reemplazar valores nulos representados como 'nan', 'null', 'none' por vacíos limpios o NaN oficial
    df = df.replace(to_replace=['nan', 'NaN', 'null', 'NULL', 'None', 'none'], value=np.nan)

    return df


def unificar_carpeta(nombre_fuente, separador_default=';', id_duplicados=None):
    """
    Busca todos los archivos en la carpeta correspondiente, los une y aplica limpieza estructural.
    """
    path_carpeta = DATA_DIR / nombre_fuente
    archivos = list(path_carpeta.glob('*.csv')) + list(path_carpeta.glob('*.xlsx'))

    print("\n" + "="*50)
    print(f"PROCESANDO Y UNIFICANDO ARCHIVOS DE: {nombre_fuente.upper()}")
    print("="*50)

    if not archivos:
        print(f"⚠️ No se encontraron archivos en: {path_carpeta}")
        return

    dfs = []
    print(f"📂 Encontrados {len(archivos)} archivos:")

    for archivo in archivos:
        print(f"  └─ Leyendo: {archivo.name}")
        try:
            if archivo.suffix == '.csv':
                try:
                    df = pd.read_csv(archivo, sep=separador_default, encoding='utf-8', low_memory=False)
                except Exception:
                    df = pd.read_csv(archivo, sep=',', encoding='utf-8-sig', low_memory=False)
            else:
                df = pd.read_excel(archivo)

            dfs.append(df)
        except Exception as e:
            print(f"  ❌ Error leyendo {archivo.name}: {e}")

    if not dfs:
        print("⚠️ No se pudieron procesar datos.")
        return

    # Unificación masiva
    df_unificado = pd.concat(dfs, ignore_index=True)
    filas_totales = len(df_unificado)

    # Limpieza estructural
    df_unificado = limpiar_y_estandarizar_df(df_unificado)

    # Deduplicación
    if id_duplicados and id_duplicados in df_unificado.columns:
        df_unificado = df_unificado.drop_duplicates(subset=[id_duplicados], keep='first')
    else:
        df_unificado = df_unificado.drop_duplicates(keep='first')

    filas_limpias = len(df_unificado)
    duplicados_eliminados = filas_totales - filas_limpias

    # Exportar archivo consolidado sin alterar la lógica de negocio
    ruta_salida = OUTPUT_DIR / f'dataset_{nombre_fuente.lower()}_unificado.csv'
    df_unificado.to_csv(ruta_salida, index=False, encoding='utf-8-sig')

    print(f"\n✅ Finalizado con éxito:")
    print(f"  • Filas totales consolidadas: {filas_totales}")
    print(f"  • Filas duplicadas omitidas: {duplicados_eliminados}")
    print(f"  • Total filas limpias guardadas: {filas_limpias}")
    print(f"📁 Guardado en: {ruta_salida}")


# ==========================================================
# EJECUCIÓN DEL PROCESO
# ==========================================================
if __name__ == '__main__':
    print("🚀 INICIANDO UNIFICACIÓN Y LIMPIEZA ESTRUCTURAL DE ARCHIVOS")

    # 1. VTEX (Ventas): Deduplica por la columna 'Order' si existe
    unificar_carpeta(nombre_fuente='VTEX', separador_default=';', id_duplicados='Order')

    # 2. META: Deduplica por filas idénticas
    unificar_carpeta(nombre_fuente='Meta', separador_default=',')

    # 3. GOOGLE: Deduplica por filas idénticas
    unificar_carpeta(nombre_fuente='Google', separador_default=',')

    print("\n" + "="*50)
    print("🎉 ¡TODOS LOS DATASETS FUERON UNIFICADOS Y LIMPIADOS EXITOSAMENTE!")
    print("="*50)

🚀 INICIANDO UNIFICACIÓN Y LIMPIEZA ESTRUCTURAL DE ARCHIVOS

PROCESANDO Y UNIFICANDO ARCHIVOS DE: VTEX
📂 Encontrados 9 archivos:
  └─ Leyendo: Reporte 30-jun-2026-31-jul-2026.csv
  └─ Leyendo: Reporte 31-dic-2025-2-jul-2025.csv
  └─ Leyendo: Reporte 1-ene-2025-1-oct-2024.csv
  └─ Leyendo: Reporte 1-jul-2025-2-ene-2025.csv
  └─ Leyendo: Reporte 30-sep-2024-1-jul-2024.csv
  └─ Leyendo: Reporte 1-ene-2026-29-jun-2026.csv
  └─ Leyendo: Reporte 30-jul-2023-1-mar-2023.csv
  └─ Leyendo: Reporte 30-jun-2024-1-ene-2024.csv
  └─ Leyendo: Reporte 31-dic-2023-31-jul-2023.csv

✅ Finalizado con éxito:
  • Filas totales consolidadas: 122328
  • Filas duplicadas omitidas: 79776
  • Total filas limpias guardadas: 42552
📁 Guardado en: datasets_procesados/dataset_vtex_unificado.csv

PROCESANDO Y UNIFICANDO ARCHIVOS DE: META
📂 Encontrados 1 archivos:
  └─ Leyendo: Ventas Meta jun-1-2023-al-jul-1-2026.csv

✅ Finalizado con éxito:
  • Filas totales consolidadas: 1116
  • Filas duplicadas omitidas: 0
  • Tota

In [3]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path('datasets_procesados')

# Configuración explícita de Pandas para NO recortar ninguna columna
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)

archivos_unificados = [
    'dataset_vtex_unificado.csv',
    'dataset_meta_unificado.csv',
    'dataset_google_unificado.csv'
]

for nombre_archivo in archivos_unificados:
    ruta_completa = OUTPUT_DIR / nombre_archivo
    
    print("\n" + "="*80)
    print(f"📄 AUDITORÍA COMPLETA: {nombre_archivo}")
    print("="*80)
    
    if not ruta_completa.exists():
        print(f"⚠️ El archivo {nombre_archivo} no existe en {OUTPUT_DIR}")
        continue
        
    df = pd.read_csv(ruta_completa, low_memory=False)
    
    filas, columnas = df.shape
    print(f"📊 Cantidad total de filas:    {filas:,}")
    print(f"📐 Cantidad total de columnas: {columnas}")
    print("-" * 80)
    
    # Construcción de la tabla resumen de TODAS las columnas
    info_cols = pd.DataFrame({
        'N°': range(1, columnas + 1),
        'Nombre de Columna': df.columns,
        'Tipo de Dato': df.dtypes.astype(str).values,
        'Valores Presentes': df.notnull().sum().values,
        'Valores Vacíos (NaN)': df.isnull().sum().values,
        '% Vacío': (df.isnull().sum().values / filas * 100).round(2)
    }).set_index('N°')
    
    # Imprimir la tabla completa de columnas sin omitir ninguna
    print(info_cols.to_string())
    print("-" * 80)

# Restablecer configuración por defecto de Pandas al terminar
pd.reset_option('display.max_columns')
pd.reset_option('display.max_rows')
pd.reset_option('display.width')


📄 AUDITORÍA COMPLETA: dataset_vtex_unificado.csv
📊 Cantidad total de filas:    42,552
📐 Cantidad total de columnas: 94
--------------------------------------------------------------------------------
                 Nombre de Columna Tipo de Dato  Valores Presentes  Valores Vacíos (NaN)  % Vacío
N°                                                                                               
1                           Origin          str              42552                     0     0.00
2                            Order          str              42552                     0     0.00
3                         Sequence        int64              42552                     0     0.00
4                    Creation Date          str              42552                     0     0.00
5                      Client Name          str              42552                     0     0.00
6                 Client Last Name          str              42552                     0     0.00
7              

In [4]:
from procesar_vtex_agrupado import generar_dataset_vtex_por_orden
generar_dataset_vtex_por_orden()


PROCESANDO VTEX: AGRUPACIÓN A NIVEL DE ORDEN ÚNICA
📂 Leyendo: dataset_vtex_unificado.csv...
📊 Total de ítems/filas iniciales: 42,552
🔄 Agrupando por 'Order'...

✅ Proceso completado exitosamente:
  • Filas iniciales (ítems): 42,552
  • Órdenes únicas finales: 42,552
📁 Guardado en: datasets_procesados/dataset_vtex_agrupado_ordenes.csv


In [23]:
import pandas as pd
from pathlib import Path

# Usamos la misma estructura de ruta que te funcionó
OUTPUT_DIR = Path('datasets_procesados')
ruta_vtex = OUTPUT_DIR / 'dataset_vtex_unificado.csv'

if ruta_vtex.exists():
    df = pd.read_csv(ruta_vtex, low_memory=False)
    
    # 1. Imprimir TODOS los registros de la columna UtmSource
    print("=== TODOS LOS RESULTADOS DE UtmSource ===")
    print(df['SKU Name'].unique())
else:
    print(f"⚠️ El archivo no existe en la ruta: {ruta_vtex.resolve()}")

=== TODOS LOS RESULTADOS DE UtmSource ===
<ArrowStringArray>
[                    'Tinte #7.13 Rubio Avellana Green Code Kit 50g',
                'Tinte #8.0 Rubio Claro Natural Green Code Kit  50g',
                    'Acondicionador Cabellos Secos Green Code 400ml',
                            'Shampoo Vegan Keratin Collagen x1000ml',
                 'Shampoo Color Intensifier Platinum SalonIn 1000ml',
                                   'Firming Pomade Vitane Men 200 g',
                                'Shampoo Liss Control SalonIn 300ml',
                               'Shampoo Rizos Perfectos Muss  400ml',
                             'Protector Labial CHAPSTICK Minions x3',
                             'Intens Mask Hydra Repair SalonIn 300g',
 ...
              'Espuma para Pies y Talones Cuarteados Deo Pies 60 ml',
                              'Desodorante en Talco Deo Pies  150 g',
                              'Tanga SPF 4 Crema Bronceador x 220ml',
   'Jabón Líquido Antiba

In [15]:
import os

print("Tu carpeta actual es:", os.getcwd())
print("Archivos/carpetas aquí:", os.listdir())

Tu carpeta actual es: /Users/richardacuna/Desktop/Analisis_TiendaCo
Archivos/carpetas aquí: ['datasets_procesados', '.DS_Store', 'LICENSE', 'requirements.txt', 'notebook.ipynb', '__pycache__', 'README.md', 'etl_procesamiento.py', '.gitignore', 'procesar_vtex_agrupado.py', 'app.py', '.git', 'Data']
